In [4]:
import datetime

import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

In [5]:
sns.set_theme(style="whitegrid", palette="deep")
task_palette = sns.color_palette("deep")
plt.rcParams["text.usetex"] = True
plt.rcParams.update({"figure.titlesize": "small"})
plt.rcParams.update({"axes.titlesize": "small"})
plt.rcParams.update({"axes.labelsize": "small"})
plt.rcParams.update({"ytick.labelsize": "small"})
plt.rcParams.update({"xtick.labelsize": "small"})
plt.rcParams.update({"legend.fontsize": "small"})

In [6]:
path = "./results/results-mcs_scale01-2026-04-06_20-41-31.csv"
df = pd.read_csv(path, sep=";")
df

,scheduler,safe_oracles,use_case,use_idlesim,unsafe_oracles,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,3,0.3,0.295,5.0,1.0,False,True,101.0,790.0,2.836362e+06
1,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,13,0.3,0.295,5.0,1.0,False,True,101.0,1921.0,5.415404e+06
2,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,12,0.3,0.300,5.0,1.0,False,True,101.0,799.0,2.319242e+06
3,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,6,0.3,0.300,5.0,1.0,False,True,22.0,1756.0,7.070575e+06
4,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,4,0.3,0.295,5.0,1.0,False,True,11.0,4193.0,1.460749e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,137,0.9,0.900,5.0,0.0,False,False,100.0,16087487.0,2.238456e+11
128,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,125,0.9,0.900,5.0,0.0,False,True,80.0,18896548.0,2.745787e+11
129,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,146,1.0,1.000,5.0,0.0,False,False,101.0,24122479.0,3.437589e+11
130,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,126,0.9,0.905,5.0,0.0,True,NaN,NaN,NaN,NaN


In [7]:
df["duration_s"] = df["duration_ns"] / 1e9

In [8]:
df["search_type"] = df["use_idlesim"].apply(lambda x: "ACBFS" if x else "BFS")

In [9]:
df = df.dropna(how="any")

In [14]:
s = df.shape[0] * [True]
df_plot = df.loc[s]
df_piv = df_plot.copy().pivot(
    index="taskset_position",
    columns="search_type",
    values=["duration_s", "visited_count", "max_period", "nbt"],
)
df_piv.columns = list(map(lambda x: "_".join(x), df_piv.columns))

In [15]:
df_piv = df_piv.dropna(how="any")

In [16]:
df_piv["Number of tasks"] = df_piv["nbt_BFS"].astype(int)
df_piv["Period max"] = df_piv["max_period_ACBFS"].astype(int)
df_piv["n"] = df_piv["nbt_BFS"].astype(int)
df_piv["T max"] = df_piv["max_period_ACBFS"].astype(int)

KeyError: 'nbt_BFS'

In [ ]:
f, ax = plt.subplots(figsize=(6, 3))

sns.scatterplot(
    data=df_piv,
    x="duration_s_BFS",
    y="duration_s_ACBFS",
    style="T max",
    hue="n",
    ax=ax,
    palette=task_palette,
)
ax.set(xscale="log", yscale="log")
# ax.axis("equal")

lims = [
    10**-5.1,
    np.max([ax.get_xlim(), ax.get_ylim()]),  # max of both axes
]

# now plot both limits against eachother
ax.plot(lims, lims, "k--", alpha=0.75, zorder=1)

ax.set(xlim=lims, ylim=lims)

ax.set_xlabel("BFS")
ax.set_ylabel("ACBFS")
ax.set_title("Execution time (s)")

# sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1.1))

f.suptitle(r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, U^{*} = 0.5$", y=1.02)

h, l = ax.get_legend_handles_labels()
ax.legend_.remove()
leg = [
    "$n$",
    "2",
    "3",
    "4",
    "5",
    "6",
    "$T^{\mathsf{max}}$",
    "10",
    "15",
    "20",
    "25",
    "30",
]
f.legend(h, leg, ncol=2, loc="upper left", bbox_to_anchor=(0.15, 0.85), framealpha=1)


x = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_BFS"])
y = np.log10(df_piv.sort_values(by="duration_s_BFS")["duration_s_ACBFS"])
m, b = np.polyfit(x, y, 1)
x_reg = np.linspace(lims, 100)
x_reg = np.log10(x_reg)
plt.plot(10**x_reg, 10 ** (m * x_reg + b), color="k", ls=":")

In [ ]:
f.savefig(
    f"./fig_rtss//{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_bfs_acbfs.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./fig_rtss/bfs_acbfs.pdf",
    bbox_inches="tight",
)

In [18]:
path = "./results/results-mcs_scale01-2026-04-06_20-41-31.csv"

df_sp = pd.read_csv(path, sep=";")
df_sp

,scheduler,safe_oracles,use_case,use_idlesim,unsafe_oracles,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,3,0.3,0.295,5.0,1.0,False,True,101.0,790.0,2.836362e+06
1,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,13,0.3,0.295,5.0,1.0,False,True,101.0,1921.0,5.415404e+06
2,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,12,0.3,0.300,5.0,1.0,False,True,101.0,799.0,2.319242e+06
3,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,6,0.3,0.300,5.0,1.0,False,True,22.0,1756.0,7.070575e+06
4,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,4,0.3,0.295,5.0,1.0,False,True,11.0,4193.0,1.460749e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,137,0.9,0.900,5.0,0.0,False,False,100.0,16087487.0,2.238456e+11
128,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,125,0.9,0.900,5.0,0.0,False,True,80.0,18896548.0,2.745787e+11
129,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,146,1.0,1.000,5.0,0.0,False,False,101.0,24122479.0,3.437589e+11
130,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,126,0.9,0.905,5.0,0.0,True,NaN,NaN,NaN,NaN


In [19]:
df_sp["taskset_file"].value_counts()

taskset_file
outputs/20260406_154058-harmonic.txt    132
Name: count, dtype: int64

In [20]:
df_sp.groupby("taskset_file")["taskset_position"].max()

taskset_file
outputs/20260406_154058-harmonic.txt    159
Name: taskset_position, dtype: int64

In [21]:
df_sp["duration_s"] = df_sp["duration_ns"] / 1e9

In [23]:
df_sp["timeout_int"] = df_sp["timeout"].astype(int)
df_sp.groupby(["nbt"])["timeout_int"].agg(["count", "sum"])

,count,sum
nbt,,
5.0,132,2


In [24]:
df_sp.groupby("nbt")["duration_s"].describe()

,count,mean,std,min,25%,50%,75%,max
nbt,,,,,,,,
5.0,130.0,20.918347,49.128407,0.002319,0.09849,1.442774,20.537645,343.758922


In [11]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_tmax_ntask.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/tmax_ntask.pdf",
    bbox_inches="tight",
)

NameError: name 'f' is not defined

In [26]:
path = "./results/results-mcs_scale01-2026-04-06_20-41-31.csv"
df_sch = pd.read_csv(path, sep=";")
df_sch

,scheduler,safe_oracles,use_case,use_idlesim,unsafe_oracles,taskset_file,taskset_position,U,Uv,nbt,EDFVD_test,timeout,is_safe,automaton_depth,visited_count,duration_ns
0,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,3,0.3,0.295,5.0,1.0,False,True,101.0,790.0,2.836362e+06
1,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,13,0.3,0.295,5.0,1.0,False,True,101.0,1921.0,5.415404e+06
2,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,12,0.3,0.300,5.0,1.0,False,True,101.0,799.0,2.319242e+06
3,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,6,0.3,0.300,5.0,1.0,False,True,22.0,1756.0,7.070575e+06
4,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,4,0.3,0.295,5.0,1.0,False,True,11.0,4193.0,1.460749e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,137,0.9,0.900,5.0,0.0,False,False,100.0,16087487.0,2.238456e+11
128,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,125,0.9,0.900,5.0,0.0,False,True,80.0,18896548.0,2.745787e+11
129,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,146,1.0,1.000,5.0,0.0,False,False,101.0,24122479.0,3.437589e+11
130,edfvd,[],"ACBFS, no oracle",True,[],outputs/20260406_154058-harmonic.txt,126,0.9,0.905,5.0,0.0,True,NaN,NaN,NaN,NaN


In [28]:
df_edfvd_test = df_sch[["taskset_position", "EDFVD_test", "U"]]
df_edfvd_test = df_edfvd_test.rename(columns={"EDFVD_test": "EDF-VD (sufficient)"})
df_edfvd_test["EDF-VD (sufficient)"] = df_edfvd_test["EDF-VD (sufficient)"].astype(bool)

In [27]:
df_sch["schedulable"] = df_sch["is_safe"]

In [29]:
d1 = df_sch[["taskset_position", "use_case", "schedulable", "U"]]
d2 = df_edfvd_test.melt(
    id_vars=["taskset_position", "U"], var_name="use_case", value_name="schedulable"
)

In [30]:
df_sched_comb = pd.concat([d1, d2], ignore_index=True)

In [ ]:
# f, ax = plt.subplots(figsize=(3.5 * 46 / 98 * 2, 3.5))
f, ax = plt.subplots(figsize=(3.5, 3.5))

col_order = ["EDF-VD (sufficient)", "EDF-VD (exact)", "LWLF (exact)"]

sns.lineplot(
    data=df_sched_comb,
    x="U",
    y="schedulable",
    hue="use_case",
    ax=ax,
    markers=True,
    errorbar=None,
    style="use_case",
    ms=9,
    hue_order=col_order,
    style_order=col_order,
)

ax.set_ylabel("Schedulability ratio")
ax.set_xlabel("Average utilisation ($U^*$)")

# f.suptitle(
#     r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
#     x=0.55,
#     # y=0.7,
# )

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, T^{\mathsf{min}} = 5, T^{\mathsf{max}} = 30, n=5$",
)

# handles, labels = ax.get_legend_handles_labels()
# ax.legend(handles=handles, labels=labels, framealpha=1, loc="lower left")
# ax.legend(
#     handles=handles,
#     labels=labels,
#     loc="upper center",
#     ncol=2,
#     framealpha=0,
#     columnspacing=0.8,
#     bbox_to_anchor=(0.5, 1.4),
# )
sns.move_legend(
    ax,
    loc="center left",
    # ncol=2,
    framealpha=1,
    # columnspacing=0.8,
    # bbox_to_anchor=(0.5, 1.8),
    title="",
)

f.tight_layout()

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()


In [ ]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_uavg.pdf",
    bbox_inches="tight",
)
f.savefig(
    "./figs/uavg.pdf",
    bbox_inches="tight",
)

In [38]:
f, ax = plt.subplots(figsize=(3.5, 3.5))

df_time = (
    df_sch.loc[:, ["U", "duration_ns"]]
    .dropna(how="any")
    .assign(duration_s=lambda d: d["duration_ns"] / 1e9)
    .sort_values("U")
)

sns.lineplot(
    data=df_time,
    x="U",
    y="duration_s",
    marker="o",
    ax=ax,
)

ax.set_ylabel("Time taken (s)")
ax.set_xlabel("Average utilisation ($U^*$)")

ax.set_title(
    r"$P_{\mathsf{HI}} = 0.5, n=5$",
)


f.tight_layout()

/home/ochremarsh/Documents/real-time-systems/mixed-criticality-graph-xp/.venv-host/lib/python3.12/site-packages/seaborn/relational.py:293: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sub_data = grouped.apply(agg, other).reset_index()


In [40]:
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_edfvd_exact_time_uavg.pdf",
    bbox_inches="tight",
)
f.savefig(
    f"./figs/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}_edfvd_exact_time_uavg.png",
    bbox_inches="tight",
)
f.savefig(
    "./figs/edfvd_exact_time_uavg.pdf",
    bbox_inches="tight",
)